# LAM → OpenAvatarChat (OAC) avatar bake — Colab (free T4)

Bake **one** avatar from **one** portrait with [aigc3d/LAM](https://github.com/aigc3d/LAM) and export the
**OAC zip** (`skin.glb`, `offset.ply`, `animation.glb`, `vertex_order.json`) that
`ints-head-gs` loads. **Not** `h5_render_data.zip`.

**Before anything:** `Runtime ▸ Change runtime type ▸ T4 GPU`.

### Run order
1. **Cell 1** (GPU + Python version).
2. **If Cell 1 says Python ≠ 3.10** → run **Cells 1b → (auto-restart) → 1c**, then continue at Cell 2.
   **If Python == 3.10** → skip 1b/1c, go straight to Cell 2.
3. **Cells 2 → 7** top to bottom.

### ⚠️ Honest feasibility (read once)
Heavy research pipeline; full setup ~30–60 min; **not yet run on live Colab** — the call-outs are the map of
where it bites.

- **FBX SDK = the critical risk.** The OAC export builds `skin.glb` via FBX SDK → Blender, and LAM ships the
  FBX SDK **only as a cp310 wheel** — it needs **Python 3.10**. Colab is often 3.11/3.12, hence the conditional
  condacolab cells (1b/1c) that give a 3.10 base.
- **CUDA-compiled deps (Cell 2)** build from source (slow; can fail).
- **Session limits** — free Colab idles ~90 min, caps ~12 h.

**Zero-setup fallback:** LAM's [ModelScope Space](https://www.modelscope.cn/studios/Damo_XR_Lab/LAM_Large_Avatar_Model)
exports the OAC zip server-side — if Colab fights you, bake there and drop the zip into `ints-head-gs`.


In [ ]:
# Cell 1 — GPU + CUDA + Python version.  SUCCESS: T4 shown, 'CUDA available: True', and a Python version.
!nvidia-smi
import sys, torch  # Colab ships torch; Cell 2 reinstalls LAM's pinned torch 2.3.0.
print('torch', torch.__version__, '| torch CUDA', torch.version.cuda, '| available', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU. Runtime > Change runtime type > T4 GPU, then Restart and run all.'
p = torch.cuda.get_device_properties(0)
print(f'GPU: {p.name} | VRAM: {p.total_memory/1e9:.1f} GB')
PYV = '%d.%d' % sys.version_info[:2]
print('Python:', PYV)
if sys.version_info[:2] == (3, 10):
    print('\u2705 Python is 3.10 — SKIP cells 1b/1c, go straight to Cell 2.')
else:
    print('\u26a0\ufe0f Python is NOT 3.10 — run cells 1b then (after the auto-restart) 1c BEFORE Cell 2,')
    print('   so the LAM cp310 FBX wheel (needed for skin.glb / the OAC zip) can install.')
# LAM-20K inference is light (~1.4s on A100) and fits T4 (~15GB). Host RAM (~12GB) is the tighter limit.


## Cells 1b – 1c — Python 3.10 via condacolab  —  **RUN THIS ONLY IF Python ≠ 3.10**
If Cell 1 reported Python 3.10, **skip both of these** and go to Cell 2.

Why: LAM's FBX SDK is a **cp310-only** wheel; `skin.glb` (and therefore the OAC zip) can't be built without
it. These cells install a conda base pinned to **Python 3.10.x**, so every later cell runs on 3.10 with no
changes.

**Sequencing (important):**
- Run **1b first** — condacolab **auto-restarts the runtime**. The cell will look like it *crashed*
  (“Your session crashed”). **That is expected.** Do **not** re-run 1b.
- After the restart, run **1c** to confirm, then continue at **Cell 2**.
- 1b must run **before** Cell 2 — it replaces Python, which would wipe pip installs done earlier.


In [ ]:
# Cell 1b — RUN ONLY IF Python != 3.10.  Installs a Python-3.10 conda base.
# ⚠️ This AUTO-RESTARTS the runtime (looks like a crash — that's normal). Do NOT re-run this cell.
#    After the restart, run Cell 1c, then go to Cell 2.
!pip install -q condacolab
import condacolab
# Pin a Miniforge whose base env is Python 3.10.x (deterministic; later cells need no 'conda run' prefix).
condacolab.install_from_url(
    'https://github.com/conda-forge/miniforge/releases/download/23.3.1-1/Miniforge3-23.3.1-1-Linux-x86_64.sh'
)


In [ ]:
# Cell 1c — RUN AFTER THE RESTART (condacolab path only).  SUCCESS: check() OK and Python is 3.10.x.
import condacolab; condacolab.check()
import sys; print('Python now:', sys.version.split()[0])
assert sys.version_info[:2] == (3, 10), 'Base is not 3.10 — see RUNBOOK "FBX on Colab".'
# GPU still attached after restart? (condacolab keeps the runtime + GPU; only Python changed.)
import torch; assert torch.cuda.is_available(), 'GPU lost after restart — Runtime > Change runtime type > T4.'
print('\u2705 on Python 3.10 with GPU — continue at Cell 2.')


## Cell 2 — clone LAM + install deps
Picks `cu121` vs `cu118` from the CUDA the runtime exposes (Colab T4 = CUDA 12 → **cu121**). Pins torch 2.3.0
and **compiles** `pytorch3d` / `diff-gaussian-rasterization` / `nvdiffrast` / `simple-knn`.

**If it fails:**
- *pytorch3d build error* → prebuilt wheel (built for pyt2.5.1; may mismatch torch 2.3.0):
  `pip install --no-index --no-cache-dir pytorch3d -f https://dl.fbaipublicfiles.com/pytorch3d/packaging/wheels/py310_cu121_pyt251/download.html`
- *nvcc / `make.sh` fails* → `!apt-get install -y cuda-toolkit-12-1`, re-run.
- *numpy 1.23 downgrade warnings* → expected; ignore unless an import breaks.


In [ ]:
# Cell 2 — clone + install.  SUCCESS: ends 'INSTALL DONE' with no red tracebacks.  (~20-40 min)
import torch, os
cu = torch.version.cuda or ''
INSTALL = 'install_cu121.sh' if cu.startswith('12') else 'install_cu118.sh'
print('runtime CUDA', cu, '-> using', INSTALL)
if not os.path.isdir('/content/LAM'):
    !git clone https://github.com/aigc3d/LAM.git /content/LAM
%cd /content/LAM
!pip install -q ninja  # speeds the source builds
!sh ./scripts/install/{INSTALL}
print('INSTALL DONE')


## Cell 3 — download weights + assets (HuggingFace)
Repos `3DAIGC/LAM-20K` and `3DAIGC/LAM-assets` are **public** — a token is optional (helps rate limits).
Paste yours in the placeholder; **do not commit it**.


In [ ]:
# Cell 3 — weights + assets.  SUCCESS: 'WEIGHTS + ASSETS OK'.
# >>> OPTIONAL: paste your HF token (else anonymous). NEVER COMMIT THIS. <<<
HF_TOKEN = ''  # e.g. 'hf_xxx'
import os
!pip install -q 'huggingface_hub==0.23.2'
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    !huggingface-cli login --token $HF_TOKEN
# assets = sample_oac (template_file.fbx + animation.glb), sample_motion, sample_input + flame-tracking models
!huggingface-cli download 3DAIGC/LAM-assets --local-dir ./tmp
!tar -xf ./tmp/LAM_assets.tar && rm ./tmp/LAM_assets.tar
!tar -xf ./tmp/thirdparty_models.tar && rm -r ./tmp/
# LAM-20K model weights
!huggingface-cli download 3DAIGC/LAM-20K --local-dir ./model_zoo/lam_models/releases/lam/lam-20k/step_045500/
assert os.path.exists('model_zoo/lam_models/releases/lam/lam-20k/step_045500/model.safetensors'), 'LAM-20K weights missing'
assert os.path.isdir('assets/sample_oac'), 'assets/sample_oac missing (needed for OAC: template_file.fbx + animation.glb)'
print('WEIGHTS + ASSETS OK')


## Cell 4 — Blender (headless) + FBX SDK
`skin.glb` is built ASCII FBX → *(FBX SDK)* binary FBX → *(Blender)* GLB, and that Blender step also writes
`vertex_order.json`. Needs **both** Blender **and** the FBX SDK. The FBX wheel is cp310 — if `import fbx` fails
here, you skipped the condacolab path (Cells 1b/1c); go back and run it.


In [ ]:
# Cell 4 — Blender + FBX SDK.  SUCCESS: 'BLENDER OK' AND 'import fbx OK'.
import os, sys
BV = 'blender-4.0.2-linux-x64'  # guide-pinned; OAC needs Blender > 4.0
if not os.path.isdir(f'/content/{BV}'):
    !wget -q https://download.blender.org/release/Blender4.0/{BV}.tar.xz -O /content/blender.tar.xz
    !tar -xf /content/blender.tar.xz -C /content/
BLENDER = f'/content/{BV}/blender'
os.environ['BLENDER'] = BLENDER
r = os.system(f'{BLENDER} --background --version')
if r != 0:  # headless Blender still needs a few X libs present
    !apt-get -qq install -y libxi6 libxxf86vm1 libxfixes3 libxrender1 libgl1 libsm6 >/dev/null
    r = os.system(f'{BLENDER} --background --version')
assert r == 0, 'Blender headless failed — check the apt libs above.'
print('BLENDER OK:', BLENDER)

print('Python:', sys.version.split()[0])
if sys.version_info[:2] == (3, 10):
    !wget -q https://virutalbuy-public.oss-cn-hangzhou.aliyuncs.com/share/aigc3d/data/LAM/fbx-2020.3.4-cp310-cp310-manylinux1_x86_64.whl -O /content/fbx.whl
    !pip install -q /content/fbx.whl
    import fbx  # noqa
    print('import fbx OK — full OAC export available.')
else:
    raise SystemExit('Python != 3.10 — the cp310 FBX wheel cannot install. Run Cells 1b/1c (condacolab) '
                     'and re-run from Cell 2, OR bake on the ModelScope Space (see RUNBOOK).')


In [ ]:
# Cell 5 — upload your fisherman portrait.  SUCCESS: prints 'Saved: assets/sample_input/fisherman.jpg'.
# Front-facing, well-lit, head-and-shoulders works best (LAM is one-shot from a single face).
from google.colab import files
import os, shutil
up = files.upload()  # choose your image
src = list(up.keys())[0]
os.makedirs('assets/sample_input', exist_ok=True)
IMG = 'assets/sample_input/fisherman.jpg'
shutil.move(src, IMG)
print('Saved:', IMG)


## Cell 6 — bake + OAC export (headless)
Writes the gradio-free runner (adapted from `app_lam.py core_fn`) and runs it: flame-tracks the image, runs
LAM-20K, exports the four OAC files, zips them with **inner folder == zip name**.

If this raises on a LAM internal (signature drift), use the **Gradio fallback** cell below.


In [ ]:
%%writefile colab_bake_oac.py
#!/usr/bin/env python3
"""
Headless LAM → OpenAvatarChat (OAC) bake — one image in, one OAC zip out.

This is the gradio-free path used by the Colab notebook (LAM_bake_oac_colab.ipynb).
It is adapted *faithfully* from `app_lam.py`'s `core_fn` OAC-export branch
(the part gated behind the "Export ZIP file for Chatting Avatar" checkbox), with
the video-rendering steps stripped out — we only need the four OAC files.

Run from inside the cloned LAM repo:
    python colab_bake_oac.py \
        --image assets/sample_input/fisherman.jpg \
        --blender_path /content/blender-4.0.2-linux-x64/blender \
        --motion auto

Output: ./output/open_avatar_chat/<image-stem>.zip containing
    <image-stem>/skin.glb, offset.ply, animation.glb, vertex_order.json
i.e. exactly LAM's OAC format (see ints-head-gs/docs/AVATAR_FORMAT.md). NOT
h5_render_data.zip.

⚠️ This orchestration mirrors LAM internals at a point in time. If LAM changes a
signature (infer_single_view / prepare_motion_seqs / save_shaped_mesh), this will
raise — fall back to the notebook's Gradio cell, which runs LAM's own code.
"""
import argparse
import os
import sys
import shutil
import zipfile
from glob import glob
from pathlib import Path


def find_motion(motion_arg: str) -> str:
    """Pick a driving motion-sequence dir (provides flame shape + render params)."""
    if motion_arg and motion_arg != "auto":
        d = f"./assets/sample_motion/export/{motion_arg}"
        assert os.path.isdir(d), f"motion not found: {d}"
        return d
    cands = sorted(glob("./assets/sample_motion/export/*/"))
    assert cands, "no sample motions under assets/sample_motion/export/ — did the assets download succeed?"
    # prefer the one LAM's own inference.sh uses, if present
    for c in cands:
        if "Look_In_My_Eyes" in c:
            return c.rstrip("/")
    return cands[0].rstrip("/")


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--image", required=True, help="input portrait, e.g. assets/sample_input/fisherman.jpg")
    ap.add_argument("--blender_path", required=True, help="path to Blender >4.0 executable")
    ap.add_argument("--motion", default="auto", help="motion seq name under assets/sample_motion/export, or 'auto'")
    args = ap.parse_args()

    assert os.path.exists(args.image), f"image not found: {args.image}"
    assert os.path.exists(args.blender_path), f"blender not found: {args.blender_path}"

    # --- env + config, copied from app_lam.launch_gradio_app -----------------
    os.environ.update({
        "APP_ENABLED": "1",
        "APP_MODEL_NAME": "./model_zoo/lam_models/releases/lam/lam-20k/step_045500/",
        "APP_INFER": "./configs/inference/lam-20k-8gpu.yaml",
        "APP_TYPE": "infer.lam",
        "NUMBA_THREADING_LAYER": "omp",
    })

    import torch
    import app_lam  # importing does NOT launch gradio (that's under __main__)
    from tools.generateARKITGLBWithBlender import generate_glb
    from lam.runners.infer.head_utils import prepare_motion_seqs, preprocess_image

    # parse_configs reads --blender_path off sys.argv; hand it a clean argv.
    sys.argv = ["colab_bake_oac.py", "--blender_path", args.blender_path]
    cfg, _ = app_lam.parse_configs()

    print("building model + flame tracking…")
    lam = app_lam._build_model(cfg)
    lam.to("cuda").eval()

    from tools.flame_tracking_single_image import FlameTrackingSingleImage
    flametracking = FlameTrackingSingleImage(
        output_dir="output/tracking",
        alignment_model_path="./model_zoo/flame_tracking_models/68_keypoints_model.pkl",
        vgghead_model_path="./model_zoo/flame_tracking_models/vgghead/vgg_heads_l.trcd",
        human_matting_path="./model_zoo/flame_tracking_models/matting/stylematte_synth.pt",
        facebox_model_path="./model_zoo/flame_tracking_models/FaceBoxesV2.pth",
        detect_iris_landmarks=False,
    )

    motion_seqs_dir = find_motion(args.motion)
    base_iid = os.path.basename(args.image).split(".")[0]
    print(f"image={args.image}  iid={base_iid}  motion={motion_seqs_dir}")

    # --- flame tracking on the input image (core_fn steps) -------------------
    tmp_dir = "output/_bake_tmp"
    os.makedirs(tmp_dir, exist_ok=True)
    image_raw = os.path.join(tmp_dir, "raw.png")
    from PIL import Image
    with Image.open(args.image).convert("RGB") as im:
        im.save(image_raw)

    assert flametracking.preprocess(image_raw) == 0, "flametracking preprocess failed"
    assert flametracking.optimize() == 0, "flametracking optimize failed"
    rc, output_dir = flametracking.export()
    assert rc == 0, "flametracking export failed"

    image_path = os.path.join(output_dir, "images/00000_00.png")
    mask_path = os.path.join(output_dir, "fg_masks/00000_00.png")

    aspect_standard = 1.0 / 1.0
    image, _, _, shape_param = preprocess_image(
        image_path, mask_path=mask_path, intr=None, pad_ratio=0, bg_color=1.0,
        max_tgt_size=None, aspect_standard=aspect_standard, enlarge_ratio=[1.0, 1.0],
        render_tgt_size=cfg.source_size, multiply=14, need_mask=True, get_shape_param=True,
    )

    src = image_path.split("/")[-3]
    driven = motion_seqs_dir.split("/")[-2]
    motion_seq = prepare_motion_seqs(
        motion_seqs_dir, None, save_root=tmp_dir, fps=30, bg_color=1.0,
        aspect_standard=aspect_standard, enlarge_ratio=[1.0, 1, 0],
        render_image_res=cfg.render_size, multiply=16, need_mask=False,
        vis_motion=False, shape_param=shape_param, test_sample=False,
        cross_id=False, src_driven=[src, driven],
    )

    # --- inference → canonical gaussians -------------------------------------
    device, dtype = "cuda", torch.float32
    motion_seq["flame_params"]["betas"] = shape_param.unsqueeze(0)
    print("running LAM inference…")
    with torch.no_grad():
        res = lam.infer_single_view(
            image.unsqueeze(0).to(device, dtype), None, None,
            render_c2ws=motion_seq["render_c2ws"].to(device),
            render_intrs=motion_seq["render_intrs"].to(device),
            render_bg_colors=motion_seq["render_bg_colors"].to(device),
            flame_params={k: v.to(device) for k, v in motion_seq["flame_params"].items()},
        )

    # --- OAC export (app_lam.py lines ~304-342, minus the video) -------------
    oac_dir = os.path.join("./output/open_avatar_chat", base_iid)
    os.makedirs(oac_dir, exist_ok=True)
    print("writing offset.ply…")
    saved_head_path = lam.renderer.flame_model.save_shaped_mesh(
        shape_param.unsqueeze(0).cuda(), fd=oac_dir,
    )
    res["cano_gs_lst"][0].save_ply(os.path.join(oac_dir, "offset.ply"), rgb2sh=False, offset2xyz=True)

    print("generating skin.glb via Blender + FBX SDK (also writes vertex_order.json)…")
    generate_glb(
        input_mesh=Path(saved_head_path),
        template_fbx=Path("./assets/sample_oac/template_file.fbx"),
        output_glb=Path(os.path.join(oac_dir, "skin.glb")),
        blender_exec=Path(cfg.blender_path),
    )
    shutil.copy("./assets/sample_oac/animation.glb", os.path.join(oac_dir, "animation.glb"))
    if os.path.exists(saved_head_path):
        os.remove(saved_head_path)

    # --- validate the 4 OAC files, then zip with inner-folder == zip-name ----
    required = ["skin.glb", "offset.ply", "animation.glb", "vertex_order.json"]
    missing = [f for f in required if not os.path.exists(os.path.join(oac_dir, f))]
    assert not missing, f"OAC export incomplete, missing: {missing} (vertex_order.json comes from generate_glb step 4)"

    out_zip = os.path.join("./output/open_avatar_chat", base_iid + ".zip")
    if os.path.exists(out_zip):
        os.remove(out_zip)
    with zipfile.ZipFile(out_zip, "w", zipfile.ZIP_DEFLATED) as z:
        for f in required:
            z.write(os.path.join(oac_dir, f), arcname=os.path.join(base_iid, f))

    print("\n✅ OAC zip ready:", os.path.abspath(out_zip))
    print("   inner folder:", base_iid, "(== zip name; matches docs/AVATAR_FORMAT.md)")
    print("   contents:", ", ".join(required))


if __name__ == "__main__":
    main()


In [ ]:
# Cell 6 (run).  SUCCESS: '\u2705 OAC zip ready: .../output/open_avatar_chat/fisherman.zip'.
!python colab_bake_oac.py --image assets/sample_input/fisherman.jpg --blender_path $BLENDER --motion auto


### Cell 6 — Gradio fallback (only if the headless runner errors)
Runs LAM's exact UI code. Open the printed public URL → upload your image → pick a driving video example →
**tick 'Export ZIP file for Chatting Avatar'** → Generate. The zip lands in `output/open_avatar_chat/`.


In [ ]:
# Optional fallback — launch LAM's gradio app with a public share link.
# !sed -i 's/demo.launch()/demo.launch(share=True)/' app_lam.py
# !python app_lam.py --blender_path $BLENDER


In [ ]:
# Cell 7 — download the OAC zip to your Mac.  SUCCESS: browser download of fisherman.zip.
from google.colab import files
import glob, os
zips = sorted(glob.glob('output/open_avatar_chat/*.zip'), key=os.path.getmtime)
assert zips, 'no OAC zip found — Cell 6 did not complete.'
print('downloading', zips[-1])
files.download(zips[-1])


---
### Verify before baking a batch
Drop the downloaded zip into `ints-head-gs`: `http://localhost:5173/?avatar=<url>` or the drag-drop zone
(validates the 4 files + folder-name rule per `docs/AVATAR_FORMAT.md`). Confirm **one** avatar renders
before spending time on more.
